# EDA: Análisis de cadena de suministro (DataCo)

**Objetivo:** conocer la calidad y estructura de los datos antes de limpiarlos.
**Fuente:** DataCo Smart Supply Chain (Kaggle). Cada fila es una línea de pedido.
**Método:** cada sección termina con una decisión que se aplica después en `scripts/02_limpiar.py`.

## Etapa 1. Carga y visión general
Se carga el archivo y se revisa su tamaño, tipos de datos y cantidad de valores por columna.

In [1]:
import pandas as pd
from pathlib import Path

# Ruta al CSV: una carpeta arriba de notebooks/, dentro de data/raw
archivo = Path.cwd().parent / "data" / "raw" / "DataCoSupplyChainDataset.csv"

# El archivo viene en formato latin-1; sin este parámetro fallaría al leerlo
df = pd.read_csv(archivo, encoding="latin-1")

# Filas y columnas (esperado: 180.519 filas y 53 columnas)
print(df.shape)
df.head()

(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [2]:
# info() muestra por columna: nombre, cantidad de valores no nulos y tipo de dato.
# Sirve para detectar columnas con datos faltantes y fechas guardadas como texto.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

## Etapa 2. Calidad de los datos
Se buscan valores nulos, filas duplicadas y columnas repetidas o innecesarias.

In [3]:
# Cuenta los valores vacíos por columna y muestra solo las que tienen alguno
nulos = df.isna().sum()
nulos[nulos > 0]

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

**Decisión:** `Product Description` (100% vacía) y `Order Zipcode` (86% vacía) no aportan información: se eliminan.

In [4]:
# Grano = qué representa una fila. Se comprueba que no haya filas repetidas
# y que Order Item Id (identificador de la línea) sea único.
print("Filas duplicadas:", df.duplicated().sum())
print("Order Item Id es único:", df["Order Item Id"].is_unique)

# Un pedido puede tener varias líneas, por eso hay menos pedidos que filas
print("Pedidos distintos:", df["Order Id"].nunique())
print("Clientes distintos:", df["Customer Id"].nunique())

Filas duplicadas: 0
Order Item Id es único: True
Pedidos distintos: 65752
Clientes distintos: 20652


**Decisión:** no hay duplicados. Cada fila es una **línea de pedido**, no un pedido. Los indicadores se calcularán por línea y se documentará en el README.

In [5]:
# Si dos columnas son idénticas, una sobra. equals() devuelve True si son iguales.
print("Customer Id = Order Customer Id:", df["Customer Id"].equals(df["Order Customer Id"]))
print("Category Id = Product Category Id:", df["Category Id"].equals(df["Product Category Id"]))
print("Product Card Id = Order Item Cardprod Id:", df["Product Card Id"].equals(df["Order Item Cardprod Id"]))

# En los montos se compara la mayor diferencia entre columnas (0 o casi 0 = iguales)
print("Dif. máx. Benefit vs Profit:", (df["Benefit per order"] - df["Order Profit Per Order"]).abs().max())
print("Dif. máx. Product Price vs Order Item Product Price:", (df["Product Price"] - df["Order Item Product Price"]).abs().max())
print("Dif. máx. Sales per customer vs Order Item Total:", (df["Sales per customer"] - df["Order Item Total"]).abs().max())

Customer Id = Order Customer Id: True
Category Id = Product Category Id: True
Product Card Id = Order Item Cardprod Id: True
Dif. máx. Benefit vs Profit: 0.0
Dif. máx. Product Price vs Order Item Product Price: 0.0
Dif. máx. Sales per customer vs Order Item Total: 0.0


In [6]:
# Columnas con datos personales de clientes
personales = ["Customer Email", "Customer Fname", "Customer Lname",
              "Customer Password", "Customer Street", "Customer Zipcode"]
print(df[personales].head(3))

# Latitud/longitud: ¿coinciden con el destino del pedido? Se compara la cantidad
# de ciudades de cliente y de destino, y dónde se concentran las coordenadas.
print(df[["Customer City", "Order City"]].nunique())
print(df["Longitude"].describe().round(1))

  Customer Email Customer Fname Customer Lname Customer Password  \
0      XXXXXXXXX          Cally       Holloway         XXXXXXXXX   
1      XXXXXXXXX          Irene           Luna         XXXXXXXXX   
2      XXXXXXXXX        Gillian      Maldonado         XXXXXXXXX   

            Customer Street  Customer Zipcode  
0  5365 Noble Nectar Island             725.0  
1          2679 Rustic Loop             725.0  
2      8510 Round Bear Gate           95125.0  
Customer City     563
Order City       3597
dtype: int64
count    180519.0
mean        -84.9
std          21.4
min        -158.0
25%         -98.4
50%         -76.8
75%         -66.4
max         115.3
Name: Longitude, dtype: float64


**Decisión:** los datos personales se eliminan por privacidad (nunca deben llegar a la base ni a GitHub). Las coordenadas se concentran en América, pero los pedidos llegan a muchas más ciudades de otros continentes: no representan el destino, se eliminan. Para mapas se usará país y región.

In [7]:
# Cantidad de pedidos por estado de entrega
print(df["Delivery Status"].value_counts())

# Cruce entre estado del pedido y estado de entrega, para entender los cancelados
pd.crosstab(df["Order Status"], df["Delivery Status"])

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64


Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Order Status,,,,
CANCELED,0,0,3692,0
CLOSED,4809,11109,0,3698
COMPLETE,14136,34199,0,11156
ON_HOLD,2413,5450,0,1941
PAYMENT_REVIEW,420,1082,0,391
PENDING,4857,11712,0,3658
PENDING_PAYMENT,9588,22922,0,7322
PROCESSING,5369,12503,0,4030
SUSPECTED_FRAUD,0,0,4062,0


**Decisión:** `Shipping canceled` (7.754 filas) ocurre solo con `CANCELED` y `SUSPECTED_FRAUD`. Se marcan como cancelados y se excluyen de los indicadores de entrega, porque esas entregas no se realizaron.

In [8]:
# Un envío es tardío si tardó más días de los programados
tardio = df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]

# Se excluyen los cancelados para calcular el % de entregas tardías
no_cancelado = ~df["Order Status"].isin(["CANCELED", "SUSPECTED_FRAUD"])
print("% entregas tardías:", round(tardio[no_cancelado].mean() * 100, 1))

# Se compara con la columna original del dataset
pd.crosstab(df["Delivery Status"], tardio)

% entregas tardías: 57.3


col_0,False,True
Delivery Status,,
Advance shipping,41592,0
Late delivery,0,98977
Shipping canceled,3331,4423
Shipping on time,32196,0


**Hallazgo:** alrededor del 57% de las entregas llega tarde. La regla "días reales > programados" coincide con `Delivery Status` en los pedidos no cancelados, así que es una definición confiable.

## Etapa 4. Montos, descuentos y fechas
Se revisan rangos de ventas, descuentos y ganancias, y se valida el periodo de datos.

In [10]:
# describe() entrega mínimo, máximo, promedio y cuartiles de cada columna
df[["Sales", "Order Item Discount Rate", "Order Profit Per Order",
    "Days for shipping (real)", "Days for shipment (scheduled)"]].describe().round(2)

,Sales,Order Item Discount Rate,Order Profit Per Order,Days for shipping (real),Days for shipment (scheduled)
count,180519.00,180519.00,180519.00,180519.00,180519.00
mean,203.77,0.10,21.97,3.50,2.93
std,132.27,0.07,104.43,1.62,1.37
min,9.99,0.00,-4274.98,0.00,0.00
25%,119.98,0.04,7.00,2.00,2.00
50%,199.92,0.10,31.52,3.00,4.00
75%,299.95,0.16,64.80,5.00,4.00
max,1999.99,0.25,911.80,6.00,4.00


In [11]:
# Precios con ruido decimal (ej: 59.99000168): hay que redondear a 2 decimales
print(df["Product Price"].unique()[:5])

# Porcentaje de líneas con ganancia negativa
print("% líneas con pérdida:", round((df["Order Profit Per Order"] < 0).mean() * 100, 1))

[327.75        59.99000168  39.99000168  50.          44.99000168]
% líneas con pérdida: 18.7


**Decisión:** los montos se redondean a 2 decimales. Las pérdidas (hasta -4.274) son reales del negocio: se conservan y se marcan con una columna `es_perdida`.

In [12]:
# Las fechas vienen como texto mes/día/año hora:minuto. Se convierten para validarlas.
fecha_pedido = pd.to_datetime(df["order date (DateOrders)"], format="%m/%d/%Y %H:%M")
fecha_envio = pd.to_datetime(df["shipping date (DateOrders)"], format="%m/%d/%Y %H:%M")

print("Periodo:", fecha_pedido.min(), "->", fecha_pedido.max())
print("Envíos anteriores al pedido:", (fecha_envio < fecha_pedido).sum())

Periodo: 2015-01-01 00:00:00 -> 2018-01-31 23:38:00
Envíos anteriores al pedido: 0


**Decisión:** las fechas se convierten a formato fecha. El periodo va de enero 2015 a enero 2018 y no hay inconsistencias.

## Resumen de decisiones

| Problema encontrado | Decisión | Dónde se aplica |
|---|---|---|
| Columnas vacías o casi vacías | Eliminar | `02_limpiar.py` |
| Datos personales | Eliminar por privacidad | `02_limpiar.py` |
| Columnas idénticas a otra | Eliminar la repetida | `02_limpiar.py` |
| Coordenadas que no corresponden al destino | Eliminar | `02_limpiar.py` |
| Montos con ruido decimal | Redondear a 2 decimales | `02_limpiar.py` |
| Fechas como texto | Convertir a fecha | `02_limpiar.py` |
| Pedidos cancelados o con fraude | Marcar y excluir de indicadores de entrega | `02_limpiar.py` |
| Pérdidas extremas | Conservar y marcar | `02_limpiar.py` |
| Cada fila es una línea de pedido | Documentar en README | `README.md` |